# Patchscopes: Using the Model to Explain Itself

**Survey Reference:** Section 9.4 - Patchscopes

## Overview

**Patchscopes** use the language model itself to decode and explain its intermediate representations. Instead of training external probes, we patch hidden states into prompts that ask the model to describe what it represents.

### The Core Idea

1. Extract hidden state $h$ from the source model
2. Create a prompt like "The meaning of X is:"
3. Patch $h$ at the position of "X"
4. Let the model generate an explanation

## Learning Objectives

1. Understand the patchscopes framework
2. Implement basic patchscopes for token prediction
3. Design prompts for different interpretation tasks
4. Evaluate the reliability of model self-explanations

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Basic Patchscopes Setup

We need:
- A **source** prompt to extract representations from
- A **target** prompt template for interpretation

In [ ]:
# Load model
model_name = "gpt2-medium"  # Larger model works better for self-interpretation
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Layers: {model.config.n_layer}")

In [ ]:
def extract_hidden_state(
    model, 
    tokenizer, 
    source_text: str, 
    layer_idx: int,
    position: int = -1,  # -1 = last position
):
    """
    Extract hidden state from source text at specified layer and position.
    """
    inputs = tokenizer(source_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    h = outputs.hidden_states[layer_idx][0, position, :]  # [hidden_size]
    return h

## 2. Implementing Patchscopes

In [ ]:
class PatchscopeHook:
    """
    Hook to patch a hidden state at a specific position.
    """
    
    def __init__(self, patch_position: int, patch_value: torch.Tensor):
        self.patch_position = patch_position
        self.patch_value = patch_value
    
    def __call__(self, module, input, output):
        # output is tuple: (hidden_states, ...)
        hidden_states = output[0]
        hidden_states[0, self.patch_position, :] = self.patch_value
        return (hidden_states,) + output[1:]


def patchscope(
    model,
    tokenizer,
    hidden_state: torch.Tensor,
    target_prompt: str,
    patch_position: int,  # Position in target prompt to patch
    patch_layer: int,     # Layer to patch at
    max_new_tokens: int = 20,
):
    """
    Run patchscope: patch hidden state and generate.
    
    Args:
        hidden_state: The hidden state to patch in [hidden_size]
        target_prompt: Prompt template with a placeholder
        patch_position: Token position to patch
        patch_layer: Which layer to apply the patch
    """
    inputs = tokenizer(target_prompt, return_tensors="pt").to(device)
    
    # Get the layer to hook
    layer_to_hook = model.transformer.h[patch_layer]
    
    # Register hook
    hook = PatchscopeHook(patch_position, hidden_state)
    handle = layer_to_hook.register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated

## 3. Token Identity Patchscope

The simplest patchscope: ask "What token does this represent?"

In [ ]:
def token_identity_patchscope(
    model,
    tokenizer,
    source_text: str,
    layer_idx: int,
):
    """
    Ask the model to identify what token a hidden state represents.
    """
    # Extract hidden state
    h = extract_hidden_state(model, tokenizer, source_text, layer_idx, position=-1)
    
    # Target prompt: "X" represents the word
    target_prompt = '"X" is the word'
    
    # Find position of "X"
    target_inputs = tokenizer(target_prompt, return_tensors="pt")
    # Typically "X" is at position 1 (after ")
    patch_position = 1
    
    # Patch at same layer we extracted from
    result = patchscope(
        model, tokenizer, h, target_prompt, 
        patch_position, layer_idx
    )
    
    return result

# Example
source = "The capital of France is Paris"
# result = token_identity_patchscope(model, tokenizer, source, layer_idx=20)
# print(result)

## 4. Attribute Extraction Patchscope

Ask about specific attributes of the entity.

In [ ]:
def attribute_patchscope(
    model,
    tokenizer,
    source_text: str,
    layer_idx: int,
    attribute_prompt: str,  # e.g., "The language spoken in X is"
):
    """
    Extract an attribute from a hidden state.
    """
    h = extract_hidden_state(model, tokenizer, source_text, layer_idx, position=-1)
    
    # Find X position in attribute prompt
    tokens = tokenizer.tokenize(attribute_prompt)
    x_position = None
    for i, tok in enumerate(tokens):
        if 'X' in tok:
            x_position = i
            break
    
    if x_position is None:
        raise ValueError("Prompt must contain 'X' as placeholder")
    
    result = patchscope(
        model, tokenizer, h, attribute_prompt,
        x_position, layer_idx
    )
    
    return result

# Example usage
# result = attribute_patchscope(
#     model, tokenizer,
#     source_text="The capital of France is Paris",
#     layer_idx=20,
#     attribute_prompt="The country of X is"
# )

## 5. Comparing Patchscopes Across Layers

See how the model's "understanding" develops across layers.

In [ ]:
def layer_comparison_patchscope(
    model,
    tokenizer,
    source_text: str,
    target_prompt: str,
    layers_to_check: list[int] = None,
):
    """
    Compare patchscope outputs across different layers.
    """
    if layers_to_check is None:
        n_layers = model.config.n_layer
        layers_to_check = [0, n_layers//4, n_layers//2, 3*n_layers//4, n_layers-1]
    
    results = {}
    for layer in layers_to_check:
        h = extract_hidden_state(model, tokenizer, source_text, layer, position=-1)
        
        # Simplified: patch at position 1 (adjust based on prompt)
        result = patchscope(
            model, tokenizer, h, target_prompt,
            patch_position=1, patch_layer=layer
        )
        results[layer] = result
    
    return results

# TODO: Compare across layers

## 6. Limitations and Reliability

Patchscopes have important caveats:

1. **Model capability:** The model must be able to articulate concepts
2. **Prompt sensitivity:** Results depend heavily on prompt choice
3. **Layer mismatch:** Patching early-layer hidden states at late layers may not work
4. **No ground truth:** Hard to verify if explanations are accurate

In [ ]:
# TODO: Evaluate reliability by comparing to known ground truth

## Key Takeaways

1. **Patchscopes** use the model itself as an interpreter
2. **No training required:** Works out of the box with capable models
3. **Flexible:** Can ask various questions about representations
4. **Layer-sensitive:** Works best when patch layer matches extraction layer
5. **Reliability varies:** Not a ground truth, but a useful signal

## Next Steps

→ **04_probing_classifiers.ipynb**: Train external classifiers to probe representations